In [5]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [6]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [7]:
# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [8]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"

In [9]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [10]:
chain = prompt | llm | output_parser

In [11]:
# Run LLM on a given article
def run_llm(headline, body):
    return chain.invoke({"headline": headline, "body": body})

## NER Model

In [12]:
import spacy
from span_marker import SpanMarkerModel

In [13]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [14]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [15]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [16]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [17]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [18]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file)

## Pipeline Entry Point

In [19]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [20]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 10 articles
raw_df = full_df.sample(5)
# raw_df = full_df
len(raw_df)


5

In [21]:
# raw_df = pd.read_csv(sample_data_path)

In [22]:
raw_df.head(10)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
3383,00000178-cfd2-dbb7-a7ff-cffad80b0000,Article,What We Know About The Suspect Who Planted Bom...,What We Know About The Suspect Who Planted Bom...,"Tim Mak, Dina Temple-Raston",NaN,National News,NaN,/national-news/2021/04/14/what-we-know-about-t...,Wed Apr 14 05:42:00 EDT 2021,TRUE,More than three months after the U.S. Capitol ...
1693,00000177-1cec-d1d4-a57f-7cfcfd4a0001,Article,Frontline Workers Miss Socializing and Camarad...,Frontline Workers Miss Socializing and Camarad...,"Arun Rath, Amanda Beland",NaN,Local News,NaN,/local-news/2021/01/19/frontline-workers-miss-...,Tue Jan 19 20:11:41 EST 2021,TRUE,"Today, the United States hit a grim milestone ..."
7033,0000017e-2647-da64-a77e-2f5700450001,Article,Omicron and masks: What you need to know to st...,Omicron and masks: What you need to know to st...,"Meghan Smith, Lisa Wardle",NaN,Science and Technology,NaN,/science-and-technology/2022/01/04/omicron-and...,Tue Jan 04 14:11:10 EST 2022,TRUE,As the omicron variant of COVID-19 surges acro...
7881,0000017f-5ccb-d6af-a77f-5eefcb2d0001,Article,War And Peace And Empathy: A Meditation,War And Peace And Empathy: A Meditation,Brian O'Donovan,NaN,Celtic,NaN,/music/celtic/2022/03/05/war-and-peace-and-emp...,Sat Mar 05 20:42:44 EST 2022,TRUE,"As we witness, in real-time, the most signific..."
10086,00000182-f27c-d5d3-a796-f2ff44d40001,Article,"Life expectancy in the U.S. continues to drop,...","Life expectancy in the U.S. continues to drop,...","Jane Greenhalgh, Selena Simmons-Duffin",NaN,National News,NaN,/national-news/2022/08/31/life-expectancy-in-t...,Wed Aug 31 01:58:00 EDT 2022,TRUE,"Life expectancy in the U.S. fell in 2021, for ..."


The ML Model honestly just needs the `id`, `header`, and `body`.

In [23]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [24]:
# For Testing Purposes Only
# df = df[:20]

In [25]:
df["llama_prediction"] = None # Add the llama_prediction

Remove Duplicates (if any)

In [26]:
duplicates = df.duplicated(subset=['hl1'])

In [27]:
print(duplicates.value_counts())

False    5
Name: count, dtype: int64


In [28]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [29]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

100%|██████████| 5/5 [00:00<00:00, 5015.91it/s]


Clean the Body and Header with Regex

In [30]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 5/5 [00:00<00:00, 4774.94it/s]


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary

In [31]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

In [32]:
# TODO: check for unwanted locations
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            return location  
    return None

In [33]:
df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)

100%|██████████| 5/5 [00:00<00:00, 2494.23it/s]


In [34]:
df["Explicit_Pass"].value_counts().head(10)

Series([], Name: count, dtype: int64)

### NER Code First Pass

In [35]:
unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [36]:
# Return the first valid facility found, or organization if none are found
def valid_facility(entities, LLMRun):
    first_org = None
    valid_org = False
    first_fac = None

    print(entities)

    for entity in entities:
        # If it's a valid facility, return it
        if (entity.label_ == "FAC"):
            if (first_fac == None): # Save first facility
                first_fac = entity.text
                print("facility", entity.text)
            if (entity.text not in unwanted_entities["FAC"]): # If facility is valid, return it
                print("valid facility", entity.text)

                return entity.text
        
        # Check for organizations in case no facilities are found
        elif (entity.label_ == "ORG"):
            if (first_org == None): # Save first organization
                first_org = entity.text
                print("org", entity.text)
                if (entity.text not in unwanted_entities["ORG"]): # Check if it's valid
                    valid_org = True
                    print("valid org", entity.text)
            elif ( (not valid_org) and entity.text not in unwanted_entities["ORG"]): # Only switch it for a valid organization
                first_org = entity.text
                valid_org = True
                print("valid org", entity.text)

    # For LLM Prediciton, can accept an organization too
    if (LLMRun):
        if (first_org != None):
            return first_org
    return None

In [37]:
# Run NER on the body of the article and return first valid facility
def predict_NER_def(text, LLMRun=False):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        return valid_facility(entities, LLMRun)
        
    except Exception as error:
        return None

In [38]:
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return predict_NER_def(article['body'])
    except Exception as error:
        print(error)
        return None

In [39]:
def batch_processing(data, batch_size=100):
    results = []

    # Split data into batches
    batches = [data[i:i + batch_size] for i in range(0, len(data), batch_size)]

    # Process each batch
    for batch in tqdm(batches, desc="Processing Batches"):
        batch_results = [explicit_filtering_NER(article) for article in batch]
        results.extend(batch_results)

    return results

In [40]:
# Run NER in batches
# df["NER_Pass_Batch"] = batch_processing(df.to_dict('records'))

Processing Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(More than three months, U.S., hundreds, Jan. 6, the night before, two, Capitol, Washington D.C., FBI, COVID 19, Nike, Air Max Speed Turf, Sometime between 30 and 30 p.m., one, the Democratic National Committee, Republican National Committee, Doug Kouns, 22 years, FBI, Sept. 11, Kouns, inch, FBI, Barry Black, FBI, 1995, Oklahoma City, FBI, Mike Nirenberg, FBI, NPR, FBI, the time leading up to Jan., U.S., Steven Sund, Congress, the following day, two, Black, NPR, 20 years, 2021, NPR)
facility Capitol
valid facility Capitol
(Today, the United States, more than 400 000, Americans, first, just about year ago, the last 11 months, Sue Algeri, Massachusetts General Hospital, Algeri, Algeri, Brittany Sheehan, MGH, COVID, the spring, Joe Biden, tomorrow, Boston, Eleanor Kaplan, 100, the spring, Kaplan, Biden, Suzanne Algeri, Brittany Sheehan, Kaplan, 18 02)
org Massachusetts General Hospital
valid org Massachusetts General Hospital
(COVID 19, Massachusetts, Joseph Allen, Healthy Buildings, Harv

Processing Batches: 100%|██████████| 1/1 [09:06<00:00, 546.73s/it]

(U.S., 2021, the second year in row, 2019, U.S., nearly 80 years, 202o, 77 years, 2021, 76.1 years, Americans, the Centers for Disease Control and Prevention, Steven Woolf, Virginia Commonwealth University, U.S., 2021, 2020, Woolf, U.S., One, 2021, American Indian, Alaskan Native, Between 2020 and 2021, almost two years, 67.1, 2020, 65.2, 2021, Woolf, Native American, COVID 19, 2021, Americans, 2020, Hispanic Americans, year, Black Americans, year, Americans, year, 2021, 76.4, Black Americans, 0.7 year, 70.8 years, Hispanic Americans, 0.2 year, 77.7 years, Asian Americans, 0.1 year, 83.5 years, Woolf, Americans, U.S., 50, COVID, 19, Donald Trump, Biden, NPR, 2021, COVID 19, U.S., the 20th century, John Haaga, Maryland Commission on Aging, this second year, the century, U.S., years, one, U.S., decades, Haaga, 50 years, 2022, NPR)
org the Centers for Disease Control and Prevention
valid org the Centers for Disease Control and Prevention


In [ ]:
df

In [ ]:
import multiprocessing as mp

print("Number of processors: ", mp.cpu_count())

def parallel_ner(articles_df, num_processes=(2)):
    articles = articles_df.to_dict('records')

    # Create a multiprocessing pool with the specified number of processes
    with mp.Pool(processes=num_processes) as pool:
        # Initialize the tqdm progress bar
        progress_bar = tqdm(total=len(articles))

        # Define a generator that updates the progress bar
        def imap_generator():
            for result in pool.imap(explicit_filtering_NER, articles):
                yield result
                progress_bar.update(1)

        # Collect all results using the generator
        results = list(imap_generator())

        progress_bar.close()
    
    return results

# Run NER in parallel
# df["NER_Pass"] = parallel_ner(df)

In [ ]:
df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)

In [ ]:
df.head(10)

### Llama Prediction

In [ ]:
#TODO: Comply with token limit of 2048 for Llama
# Run the LLM model on the articles that haven't been tagged with a location yet. Then run NER on the LLM prediction
def predict_llama(article):
    try:
        # If the article does not have an explicit location or NER location, run LLM
        if (article['Explicit_Pass'] != None or article['NER_Pass'] != None):
            print(f"Has location from title or NER: {article['hl1']}")
            return None
        else:
            llama_prediction = run_llm(article['hl1'], article['body'])
            print(llama_prediction)
            return predict_NER_def(llama_prediction, True)
    except Exception as error:
        print(error)
        return None

In [ ]:
df['NER_Prediction'] = df.progress_apply(predict_llama, axis=1)

In [ ]:
df.head(10)

In [ ]:
## TODO: DELETE AFTER POPULATING THE UNWANTED ENTITIES CACHE
# unwanted_entities = {
#     'FAC': ['Boston'],
#     'ORG': ['New York Times'],
#     'LOC': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
#     'GPE': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
# }

# save_cache_to_file(unwanted_entities, unwanted_entities_path)

Extract locations from the most specific pass

In [ ]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    for key in ['Explicit_Pass', 'NER_Pass', 'NER_Prediction']:
        location = article.get(key)
        if location is not None:
            return location
    return None

In [ ]:
df['Locations'] = df.progress_apply(extractLocations, axis=1)

In [ ]:
df.head(10)

In [ ]:
df

## Get the coordinates

In [ ]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [ ]:
# Get the coordinates of the location
def getCoordinates(location): # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [ ]:
df['Coordinates'] = df['Locations'].progress_apply(getCoordinates)

In [ ]:
df.head(10)

## Geocode locations

In [ ]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [ ]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"],
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County
    

In [ ]:
df[['Tracts', 'County']] = df.progress_apply(lambda row: pd.Series(geocode(row['Locations'])), axis=1)
df

In [ ]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

In [ ]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

In [ ]:
print(df['NER_Prediction'].value_counts().sum())
df['NER_Prediction'].value_counts()

In [ ]:
df

In [ ]:
df.head(10)

In [ ]:
len(df)

In [ ]:
df = df.dropna(subset=["Tracts", "County"]) # Clean those that don't have a Tract or a County

In [ ]:
print(len(df))
df.head(10)

## Topic Modeling

In [ ]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [ ]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [ ]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [ ]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns